In [ ]:
!pip install unsloth trl datasets

In [ ]:
==============================================================================

P2S GPU TRAINING NOTEBOOK CELL (GOOGLE COLAB / A100 80GB)

==============================================================================

Model: Qwen3.5-9B (Unsloth 4-bit NF4)

Strategy: Supervised Fine-Tuning (SFT) with Response-Only Masking & NEFTune

Output: LoRA Adapters + Merged 16-bit & 4-bit Safetensors + HF Hub Export

==============================================================================

import os
import gc
import json
import time
import torch
from pathlib import Path

── 0. CONFIGURATION & PATHS ──────────────────────────────────────────────────

RESUME_FROM_CHECKPOINT = False  # Set True to resume from a saved checkpoint step

Save outputs to Colab local disk (high-speed NVMe) to prevent Drive quota crashes

SAVE_DIR = "/content/p2s-outputs"
CKPT_DIR = f"{SAVE_DIR}/qwen35-9b-checkpoints"
FINAL_FILE = "final_training_dataset.jsonl"
MAX_SEQ_LENGTH = 24576  # 24.5k context budget for A100 80GB

Hugging Face Hub Configuration (Set PUSH_TO_HUB = True to auto-upload)

PUSH_TO_HUB = False
HF_TOKEN = os.environ.get("HF_TOKEN", "your_hf_token_here")
HF_USER = "your-username"
HF_REPO_LORA = f"{HF_USER}/qwen35-9b-p2s-lora"
HF_REPO_16BIT = f"{HF_USER}/qwen35-9b-p2s-merged-16bit"
HF_REPO_4BIT = f"{HF_USER}/qwen35-9b-p2s-merged-4bit"

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

── 1. IMPORTS ────────────────────────────────────────────────────────────────

print("[INFO] Importing Unsloth, Transformers, and TRL dependencies...")
from unsloth import FastVisionModel
from unsloth.chat_templates import train_on_responses_only
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

Verify dataset presence

if not os.path.exists(FINAL_FILE):
raise FileNotFoundError(
f"Missing '{FINAL_FILE}'! Run 'p2s prepare-dataset' first or upload the dataset file to Colab."
)

── 2. MODEL & LORA INITIALIZATION ───────────────────────────────────────────

print(f"[INFO] Loading Qwen3.5-9B via FastVisionModel (max_seq={MAX_SEQ_LENGTH})...")

model, processor = FastVisionModel.from_pretrained(
model_name="unsloth/Qwen3.5-9B",
max_seq_length=MAX_SEQ_LENGTH,
dtype=None,
load_in_4bit=True,
)

model = FastVisionModel.get_peft_model(
model,
finetune_vision_layers=False,      # Exclude vision layers from pure-text API task
finetune_language_layers=True,
finetune_attention_modules=True,
finetune_mlp_modules=True,
r=32,
target_modules=[
# Attention + MLP - core security reasoning capacity
"q_proj", "k_proj", "v_proj", "o_proj",
"gate_proj", "up_proj", "down_proj",
# Output head + embeddings - tunes token probabilities for HTTP status code
# tokens (400, 401, 500) and OCLI command flags for zero-shot generalisation
"lm_head", "embed_tokens",
],
lora_alpha=64,
# lora_dropout=0.05: regularises LoRA adapters on lm_head/embed_tokens
# preventing memorisation of static UUID/JWT surface strings
lora_dropout=0.05,
bias="none",
use_gradient_checkpointing="unsloth",
random_state=3407,
max_seq_length=MAX_SEQ_LENGTH,
)

── 3. IMAGE PROCESSOR PATCH (VL COLLATOR BYPASS) ────────────────────────────

Qwen3.5-9B is a Vision-Language model, so processor.image_processor exists.

Temporarily hide it during SFTTrainer init to force text-only collation.

_original_has_image_processor = hasattr(processor, "image_processor")
if _original_has_image_processor:
_saved_ip = processor.image_processor
del processor.image_processor
print("[PATCH] image_processor hidden for trainer init → VL collator bypassed.")

── 4. DATASET LOADING & TOKEN LENGTH SCAN ───────────────────────────────────

print(f"[INFO] Loading training dataset from '{FINAL_FILE}'...")
dataset = load_dataset("json", data_files=FINAL_FILE, split="train")

def apply_template(examples):
return {
"text": [
processor.apply_chat_template(
convo, tokenize=False, add_generation_prompt=False
)
for convo in examples["messages"]
]
}

dataset = dataset.map(apply_template, batched=True, remove_columns=dataset.column_names)
train_ds = dataset

print("[INFO] Scanning full dataset token lengths...")
sample_lengths = [len(processor.tokenizer.encode(ex)) for ex in train_ds["text"]]
n = len(sample_lengths)
sl_sorted = sorted(sample_lengths)

print("=" * 60)
print("  DATASET TOKEN LENGTH DISTRIBUTION SUMMARY")
print("=" * 60)
print(f"  Train Samples : {n:,} (Full corpus, no eval split)")
print(f"  P50 Length    : {sl_sorted[n // 2]:,} tokens")
print(f"  P90 Length    : {sl_sorted[int(n * 0.90)]:,} tokens")
print(f"  P99 Length    : {sl_sorted[int(n * 0.99)]:,} tokens")
print(f"  Max Length    : {sl_sorted[-1]:,} tokens")

n_over = sum(1 for l in sample_lengths if l > MAX_SEQ_LENGTH)
if n_over:
print(f"  [WARN] {n_over} samples exceed MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}!")
else:
print(f"  [INFO] All samples fit comfortably within MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}")
print("=" * 60)

── 5. TRAINER INITIALIZATION & RESPONSE-ONLY MASKING ────────────────────────

print("[INFO] Initializing SFT Trainer...")

trainer = SFTTrainer(
model=model,
tokenizer=processor,
train_dataset=train_ds,
dataset_text_field="text",
max_seq_length=MAX_SEQ_LENGTH,
dataset_num_proc=2,
packing=False,
neftune_noise_alpha=5.0,  # Prevents memorisation of oversampled Golden UUIDs
args=SFTConfig(
per_device_train_batch_size=1,
gradient_accumulation_steps=4,
warmup_steps=50,
num_train_epochs=6,
learning_rate=2e-4,
bf16=True,
fp16=False,
logging_steps=10,
optim="adamw_8bit",
weight_decay=0.01,
lr_scheduler_type="cosine_with_restarts",
lr_scheduler_kwargs={"num_cycles": 6},  # 1 restart cycle per epoch
seed=3407,

    # Checkpointing
    output_dir=CKPT_DIR,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    resume_from_checkpoint=RESUME_FROM_CHECKPOINT,

    report_to="none",
    remove_unused_columns=False,
    max_grad_norm=1.0,
),

)

Apply response-only masking (masks system prompt + user state history with -100)

trainer = train_on_responses_only(
trainer,
instruction_part="<|im_start|>user\n",
response_part="<|im_start|>assistant\n",
)
print("[INFO] Response-only masking applied via train_on_responses_only.")

Verify masking ratio on the first batch

_batch = next(iter(trainer.get_train_dataloader()))
_labels = _batch["labels"][0]
_masked = (_labels == -100).sum().item()
_total = len(_labels)
print(f"[INFO] Masking Check: {_masked}/{_total} tokens masked "
f"({100 * _masked / _total:.1f}% prompt, {100 * (_total - _masked) / _total:.1f}% response receives loss)")
del _batch, _labels

── 6. RESTORE IMAGE PROCESSOR ───────────────────────────────────────────────

Restore image_processor so Unsloth checkpoint saves do not raise AttributeError

if _original_has_image_processor:
processor.image_processor = _saved_ip
print("[PATCH] image_processor restored → checkpoint saves enabled.")

── 7. PRE-TRAIN HEALTH PROBE ────────────────────────────────────────────────

gpu_stats = torch.cuda.get_device_properties(0)
print(f"[GPU] {gpu_stats.name} | Total VRAM = {round(gpu_stats.total_memory / 1024**3, 1)} GB")

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model.train()
_probe = {
"input_ids": torch.ones((1, 512), dtype=torch.long, device="cuda"),
"attention_mask": torch.ones((1, 512), dtype=torch.long, device="cuda"),
"labels": torch.ones((1, 512), dtype=torch.long, device="cuda"),
}
t0 = time.time()
with torch.amp.autocast("cuda", dtype=torch.bfloat16):
_out = model(**_probe)
print(f"[PROBE] Forward pass : {time.time() - t0:.2f}s | loss = {_out.loss.item():.4f}")
_out.loss.backward()
print(f"[PROBE] Backward pass: {time.time() - t0:.2f}s → Model healthy. Launching trainer.")
del _out, _probe
gc.collect()
torch.cuda.empty_cache()

── 8. EXECUTE TRAINING LOOP ─────────────────────────────────────────────────

if RESUME_FROM_CHECKPOINT:
print(f"[INFO] Resuming training from checkpoint in '{CKPT_DIR}'...")

trainer_stats = trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)

print("\n" + "=" * 60)
print("  TRAINING COMPLETED")
print("=" * 60)
print(f"  Peak Reserved VRAM : {round(torch.cuda.max_memory_reserved() / 1024**3, 2)} GB")
print(f"  Training Runtime   : {trainer_stats.metrics['train_runtime']:.0f}s "
f"({trainer_stats.metrics['train_runtime'] / 3600:.2f} hours)")
print(f"  Final Loss         : {trainer_stats.metrics['train_loss']:.4f}")
print("=" * 60 + "\n")

── 9. LOCAL DISK SAVING & HUGGING FACE HUB EXPORT ───────────────────────────

lora_save_path = f"{SAVE_DIR}/qwen35-9b-p2s-lora"
merged_16bit_path = f"{SAVE_DIR}/qwen35-9b-p2s-merged-16bit"
merged_4bit_path = f"{SAVE_DIR}/qwen35-9b-p2s-merged-4bit"

print(f"[INFO] Saving LoRA adapters locally to '{lora_save_path}'...")
model.save_pretrained(lora_save_path)
processor.save_pretrained(lora_save_path)

if PUSH_TO_HUB:
print(f"[HUB] Pushing LoRA adapters to Hugging Face Hub: {HF_REPO_LORA}...")
model.push_to_hub(HF_REPO_LORA, token=HF_TOKEN)
processor.push_to_hub(HF_REPO_LORA, token=HF_TOKEN)

Merged 16-Bit Export

print(f"[INFO] Merging weights to 16-bit safetensors ('{merged_16bit_path}')...")
try:
model.save_pretrained_merged(merged_16bit_path, processor, save_method="merged_16bit")
if PUSH_TO_HUB:
print(f"[HUB] Pushing merged 16-bit model to Hugging Face Hub: {HF_REPO_16BIT}...")
model.push_to_hub_merged(HF_REPO_16BIT, processor, save_method="merged_16bit", token=HF_TOKEN)
except Exception as e:
print(f"[WARN] 16-bit merge failed or skipped: {e}")

Merged 4-Bit Export

print(f"[INFO] Merging weights to 4-bit safetensors ('{merged_4bit_path}')...")
try:
model.save_pretrained_merged(merged_4bit_path, processor, save_method="merged_4bit_forced")
if PUSH_TO_HUB:
print(f"[HUB] Pushing merged 4-bit model to Hugging Face Hub: {HF_REPO_4BIT}...")
model.push_to_hub_merged(HF_REPO_4BIT, processor, save_method="merged_4bit_forced", token=HF_TOKEN)
except Exception as e:
print(f"[WARN] 4-bit merge failed or skipped: {e}")

── 10. SAVE VERIFICATION & SUMMARY REPORT ───────────────────────────────────

def verify_outputs(save_dir):
save_path = Path(save_dir)
print("\n" + "=" * 60)
print("  SAVE VERIFICATION REPORT")
print("=" * 60)
all_ok = True
for folder_name in ["qwen35-9b-p2s-lora", "qwen35-9b-p2s-merged-16bit", "qwen35-9b-p2s-merged-4bit"]:
d = save_path / folder_name
if d.exists():
files = [f for f in d.rglob("*") if f.is_file()]
total_mb = sum(f.stat().st_size for f in files) / 1e6
print(f"  [✓] {folder_name}/ - {len(files)} files, {total_mb:.1f} MB")
else:
print(f"  [✗] {folder_name}/ - MISSING or NOT CREATED")
if folder_name == "qwen35-9b-p2s-lora":
all_ok = False
print("=" * 60 + "\n")
return all_ok

ok = verify_outputs(SAVE_DIR)

if ok:
print("[SUCCESS] All essential model outputs saved successfully.")
else:
print("[WARN] Some optional merged outputs failed, but LoRA adapters are intact.")